In [ ]:
import random
import os
import numpy as np
import torch

def seed_everything(seed=42):
    # 1. Base Python & OS environment
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

    # 2. PyTorch (CPU and GPU)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # For multi-GPU setups

    # 3. Force CUDA deterministic algorithms
    # Note: This ensures identical results but can slightly reduce execution speed
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Call it before loading any models or data
seed_everything(42)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install faster-whisper jiwer speechbrain librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 56.8 MB/s eta 0:00:00


In [ ]:
from faster_whisper import WhisperModel
from jiwer import wer, cer
from pathlib import Path
from tqdm import tqdm

import pandas as pd
import re
import string
import numpy as np
import random
import torch

# ==================================================
# DETERMINISM
# ==================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ==================================================
# TEXT NORMALIZATION
# ==================================================

def normalize_text(text):

    text = text.lower()

    text = text.replace("--", " ")
    text = text.replace("—", " ")
    text = text.replace("–", " ")
    text = text.replace("-", " ")

    text = text.translate(
        str.maketrans(
            "",
            "",
            string.punctuation
        )
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

# ==================================================
# LOAD WHISPER
# ==================================================

print("Loading Whisper Large-v3...")

asr_model = WhisperModel(
    "large-v3",
    device="cuda",
    compute_type="float16"
)

print("Whisper loaded.")

# ==================================================
# TRANSCRIBE
# ==================================================

def transcribe_audio(audio_path):

    segments, _ = asr_model.transcribe(
        str(audio_path),
        language="en",
        beam_size=5
    )

    return " ".join(
        seg.text.strip()
        for seg in segments
    ).strip()

# ==================================================
# GET REFERENCE TEXT
# ==================================================

def get_reference_text(
    stem,
    reference_dir
):

    txt_file = (
        reference_dir /
        f"{stem}.txt"
    )

    if txt_file.exists():

        with open(
            txt_file,
            "r",
            encoding="utf-8"
        ) as f:

            return f.read().strip()

    wav_file = (
        reference_dir /
        f"{stem}.wav"
    )

    if wav_file.exists():

        return transcribe_audio(
            wav_file
        )

    return None

# ==================================================
# MAIN FUNCTION
# ==================================================

def evaluate_asr(
    reference_dir,
    audio_dir,
    output_csv
):

    reference_dir = Path(
        reference_dir
    )

    audio_dir = Path(
        audio_dir
    )

    audio_files = sorted(
        audio_dir.glob("*.wav")
    )

    print(
        f"\nFound {len(audio_files)} files"
    )

    results = []

    for audio_file in tqdm(
        audio_files,
        desc="Processing"
    ):

        gt_text = get_reference_text(
            audio_file.stem,
            reference_dir
        )

        if gt_text is None:

            print(
                f"Missing reference for "
                f"{audio_file.name}"
            )

            continue

        pred_text = transcribe_audio(
            audio_file
        )

        gt_norm = normalize_text(
            gt_text
        )

        pred_norm = normalize_text(
            pred_text
        )

        sample_wer = wer(
            gt_norm,
            pred_norm
        )

        sample_cer = cer(
            gt_norm,
            pred_norm
        )

        results.append({

            "filename":
                audio_file.name,

            "ground_truth":
                gt_text,

            "prediction":
                pred_text,

            "ground_truth_normalized":
                gt_norm,

            "prediction_normalized":
                pred_norm,

            "wer":
                sample_wer,

            "cer":
                sample_cer
        })

    # ==================================================
    # SAVE CSV
    # ==================================================

    df = pd.DataFrame(
        results
    )

    df.to_csv(
        output_csv,
        index=False
    )

    print(
        f"\nSaved CSV:\n{output_csv}"
    )

    # ==================================================
    # SUMMARY
    # ==================================================

    print("\n")
    print("=" * 80)
    print("SUMMARY")
    print("=" * 80)

    print(
        f"Files processed: "
        f"{len(df)}"
    )

    print("\n----- WER -----")

    print(
        f"Mean   : "
        f"{df['wer'].mean():.4f}"
    )

    print(
        f"Median : "
        f"{df['wer'].median():.4f}"
    )

    print(
        f"Std    : "
        f"{df['wer'].std():.4f}"
    )

    print(
        f"Min    : "
        f"{df['wer'].min():.4f}"
    )

    print(
        f"Max    : "
        f"{df['wer'].max():.4f}"
    )

    print("\n----- CER -----")

    print(
        f"Mean   : "
        f"{df['cer'].mean():.4f}"
    )

    print(
        f"Median : "
        f"{df['cer'].median():.4f}"
    )

    print(
        f"Std    : "
        f"{df['cer'].std():.4f}"
    )

    print(
        f"Min    : "
        f"{df['cer'].min():.4f}"
    )

    print(
        f"Max    : "
        f"{df['cer'].max():.4f}"
    )

    print("\n----- QUALITY -----")

    print(
        f"Perfect WER: "
        f"{(df['wer'] == 0).mean() * 100:.2f}%"
    )

    print(
        f"WER <= 0.10: "
        f"{(df['wer'] <= 0.10).mean() * 100:.2f}%"
    )

    print(
        f"WER <= 0.20: "
        f"{(df['wer'] <= 0.20).mean() * 100:.2f}%"
    )

    print(
        f"CER <= 0.10: "
        f"{(df['cer'] <= 0.10).mean() * 100:.2f}%"
    )

    # ==================================================
    # WORST SAMPLES
    # ==================================================

    print(
        "\n========== TOP 10 WORST WER ==========\n"
    )

    worst = df.sort_values(
        "wer",
        ascending=False
    )

    for _, row in worst.head(10).iterrows():

        print("=" * 80)

        print(
            row["filename"]
        )

        print(
            f"WER: {row['wer']:.4f}"
        )

        print(
            f"CER: {row['cer']:.4f}"
        )

        print("\nGROUND TRUTH:")
        print(
            row["ground_truth"]
        )

        print("\nPREDICTION:")
        print(
            row["prediction"]
        )

    return df

Loading Whisper Large-v3...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Whisper loaded.


In [ ]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/test_16k/default",
    output_csv="default_metrics.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [03:35<00:00,  1.86it/s]


Saved CSV:
default_metrics.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.0268
Median : 0.0000
Std    : 0.0665
Min    : 0.0000
Max    : 0.5714

----- CER -----
Mean   : 0.0088
Median : 0.0000
Std    : 0.0257
Min    : 0.0000
Max    : 0.2857

----- QUALITY -----
Perfect WER: 73.50%
WER <= 0.10: 91.25%
WER <= 0.20: 97.25%
CER <= 0.10: 98.50%

========== TOP 10 WORST WER ==========

5322_7680_000028_000001.wav
WER: 0.5714
CER: 0.1304

GROUND TRUTH:
Bigger than ours?' asked Lukashka good-naturedly.

PREDICTION:
Bigger than ours? Ask Bluekoshka goodnaturedly.
6209_34599_000003_000000.wav
WER: 0.4286
CER: 0.2857

GROUND TRUTH:
A BURDEN MAKES A ROUGH ROAD ROUGHER.

PREDICTION:
A Burton makes a rough rotive rotor.
6209_34601_000042_000002.wav
WER: 0.4000
CER: 0.0606

GROUND TRUTH:
The savoury odour was perceptible.

PREDICTION:
The savory odor was perceptible.
7800_283493_000070_000000.wav
WER: 0.4000
CER: 0.1667

GROUND TRUTH:
"Got everything now?" asked Bluff.

PREDICTION:
Go

,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,Since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,0.000000,0.000000
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...","For the same reason, no attention has been giv...",for the same reason no attention has been give...,for the same reason no attention has been give...,0.000000,0.000000
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,Readers of these pages will find nothing about...,readers of these pages will find nothing about...,readers of these pages will find nothing about...,0.000000,0.000000
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",Part of Chapter 4 has already appeared in the ...,part of chapter iv has already appeared in the...,part of chapter 4 has already appeared in the ...,0.040816,0.018809
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,If Mrs. Shaw had guessed at the real reason wh...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed at the real reason why...,0.028571,0.010526
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...","Does he so? said the queen, now comprehending ...",does he so said the queen now comprehending all,does he so said the queen now comprehending all,0.000000,0.000000
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",Then if you will promise to-night to keep the ...,then if you will promise to night to keep the ...,then if you will promise to night to keep the ...,0.000000,0.000000
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...","What have I done to thee, that thou shouldst f...",what have i done to thee that thou shouldst fo...,what have i done to thee that thou shouldst fo...,0.037037,0.007407
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","Then he rose up, dressed himself hastily, and ...",then he rose up dressed himself hastily and we...,then he rose up dressed himself hastily and we...,0.000000,0.000000


In [ ]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/test_16k/vc_jennifer",
    output_csv="/content/vc_jennifer_results.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [04:59<00:00,  1.34it/s]


Saved CSV:
/content/vc_jennifer_results.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.0402
Median : 0.0000
Std    : 0.1190
Min    : 0.0000
Max    : 1.5588

----- CER -----
Mean   : 0.0185
Median : 0.0000
Std    : 0.0857
Min    : 0.0000
Max    : 1.4945

----- QUALITY -----
Perfect WER: 68.25%
WER <= 0.10: 88.50%
WER <= 0.20: 95.50%
CER <= 0.10: 97.00%

========== TOP 10 WORST WER ==========

6209_34601_000058_000000.wav
WER: 1.5588
CER: 1.4945

GROUND TRUTH:
The man kicked the stool forward and made the little boy sit down, again shoving him by the shoulders; then he pointed with his finger to the porringer which was smoking upon the stove.

PREDICTION:
the man kicked the stool forward and made the little boy sit down again shoving him by the shoulders then he pointed with his finger to the porringer which was smoking upon the stove he looked overwhelmed then he bowed down and quietly cried I will know who I love when I will hold you close and you will have knowledge e

,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,0.000000,0.000000
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...",for the same reason no attention has been give...,for the same reason no attention has been give...,for the same reason no attention has been give...,0.000000,0.000000
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,Readers of these pages will find nothing about...,readers of these pages will find nothing about...,readers of these pages will find nothing about...,0.000000,0.000000
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",part of chapter four has already appeared in t...,part of chapter iv has already appeared in the...,part of chapter four has already appeared in t...,0.040816,0.025078
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed at the real reason why...,0.028571,0.010526
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...","Does he so? said the queen, now comprehending ...",does he so said the queen now comprehending all,does he so said the queen now comprehending all,0.000000,0.000000
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",than if you will promise to-night to keep the ...,then if you will promise to night to keep the ...,than if you will promise to night to keep the ...,0.029412,0.006024
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...",what have i done to thee that thou shouldst fo...,what have i done to thee that thou shouldst fo...,what have i done to thee that thou shouldst fo...,0.000000,0.000000
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","Then he rose up, dressed himself hastily, and ...",then he rose up dressed himself hastily and we...,then he rose up dressed himself hastily and we...,0.000000,0.000000


In [ ]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/evc_test_16k/jennifer/jennifer_evc_angry",
    output_csv="evc_jennifer_angry_metrics.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [03:29<00:00,  1.91it/s]


Saved CSV:
evc_jennifer_angry_metrics.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.1170
Median : 0.0769
Std    : 0.1514
Min    : 0.0000
Max    : 1.0000

----- CER -----
Mean   : 0.0598
Median : 0.0354
Std    : 0.0827
Min    : 0.0000
Max    : 0.6538

----- QUALITY -----
Perfect WER: 28.75%
WER <= 0.10: 61.00%
WER <= 0.20: 83.50%
CER <= 0.10: 81.50%

========== TOP 10 WORST WER ==========

6209_34599_000018_000002.wav
WER: 1.0000
CER: 0.3478

GROUND TRUTH:
Roofs--dwellings--shelter!

PREDICTION:
It proves dueling's shelter.
5322_7680_000061_000006.wav
WER: 1.0000
CER: 0.6538

GROUND TRUTH:
'Eh, humbug!' thought Lukashka.

PREDICTION:
airhomburg.locasca
5322_7680_000028_000001.wav
WER: 0.8571
CER: 0.4348

GROUND TRUTH:
Bigger than ours?' asked Lukashka good-naturedly.

PREDICTION:
a bicker than ours, ass-lacash-cat-ganitrably.
5322_7680_000023_000000.wav
WER: 0.8571
CER: 0.5385

GROUND TRUTH:
'How--don't want it?' Lukashka said, laughing.

PREDICTION:
Now how? Don't wan

,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,Since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,0.031746,0.015723
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...","To the same reason, no attention has been give...",for the same reason no attention has been give...,to the same reason no attention has been given...,0.046512,0.010381
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,Rears of these pages will find nothing about t...,readers of these pages will find nothing about...,rears of these pages will find nothing about t...,0.043478,0.012658
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",The part of Chapter 4 has already appeared in ...,part of chapter iv has already appeared in the...,the part of chapter 4 has already appeared in ...,0.163265,0.090909
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,"If Mrs. Shaw had nested at the real, real reas...",if mrs shaw had guessed at the real reason why...,if mrs shaw had nested at the real real reason...,0.128571,0.086842
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...","Does he so? said the queen, now comprehending ...",does he so said the queen now comprehending all,does he so said the queen now comprehending all,0.000000,0.000000
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",than if you will promise tonight to keep the o...,then if you will promise to night to keep the ...,than if you will promise tonight to keep the o...,0.176471,0.048193
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...","What have I done to thee, that thou shouldst f...",what have i done to thee that thou shouldst fo...,what have i done to thee that thou shouldst fo...,0.000000,0.000000
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","that then he rose up, dressed himself hastily,...",then he rose up dressed himself hastily and we...,that then he rose up dressed himself hastily a...,0.076923,0.076923


In [ ]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/evc_test_16k/jennifer/jennifer_evc_sad",
    output_csv="evc_jennifer_sad_metrics.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [03:25<00:00,  1.95it/s]


Saved CSV:
evc_jennifer_sad_metrics.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.1012
Median : 0.0645
Std    : 0.1332
Min    : 0.0000
Max    : 1.0000

----- CER -----
Mean   : 0.0502
Median : 0.0298
Std    : 0.0780
Min    : 0.0000
Max    : 0.8773

----- QUALITY -----
Perfect WER: 36.50%
WER <= 0.10: 61.75%
WER <= 0.20: 85.25%
CER <= 0.10: 86.25%

========== TOP 10 WORST WER ==========

5322_7680_000061_000006.wav
WER: 1.0000
CER: 0.5385

GROUND TRUTH:
'Eh, humbug!' thought Lukashka.

PREDICTION:
earhumberg.loukaska
250_142276_000005_000004.wav
WER: 0.9250
CER: 0.8773

GROUND TRUTH:
But he had the same large, soft eyes as his daughter,--eyes which moved slowly and almost grandly round in their orbits, and were well veiled by their transparent white eyelids. Margaret was more like him than like her mother.

PREDICTION:
like hand and like her nother.
5322_7680_000015_000000.wav
WER: 0.8571
CER: 0.2941

GROUND TRUTH:
'Every one...' and Luke swayed his head.

PREDICTION:


,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,Since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,0.047619,0.025157
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...",for the same reason no attention has been give...,for the same reason no attention has been give...,for the same reason no attention has been give...,0.046512,0.034602
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,Readers of these pages will find the thing abo...,readers of these pages will find nothing about...,readers of these pages will find the thing abo...,0.173913,0.037975
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",part of chapter four has already appeared in t...,part of chapter iv has already appeared in the...,part of chapter four has already appeared in t...,0.122449,0.065831
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,if massa shaw had nested the real reason why m...,if mrs shaw had guessed at the real reason why...,if massa shaw had nested the real reason why m...,0.071429,0.055263
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...","""'Does he so?' said the queen, now comprehendi...",does he so said the queen now comprehending all,does he so said the queen now comprehending all,0.000000,0.000000
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",than if you will promise to-night to keep the ...,then if you will promise to night to keep the ...,than if you will promise to night to keep the ...,0.117647,0.042169
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...","What have I done to thee, that thou shouldst f...",what have i done to thee that thou shouldst fo...,what have i done to thee that thou shouldst fo...,0.074074,0.029630
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","Then he rose up, dressed himself hastily, and ...",then he rose up dressed himself hastily and we...,then he rose up dressed himself hastily and we...,0.000000,0.000000


In [ ]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/test_16k/vc_david",
    output_csv="/content/vc_david_results.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [03:30<00:00,  1.90it/s]


Saved CSV:
/content/vc_david_results.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.0451
Median : 0.0000
Std    : 0.0916
Min    : 0.0000
Max    : 1.0000

----- CER -----
Mean   : 0.0172
Median : 0.0000
Std    : 0.0396
Min    : 0.0000
Max    : 0.4615

----- QUALITY -----
Perfect WER: 59.25%
WER <= 0.10: 85.75%
WER <= 0.20: 95.75%
CER <= 0.10: 97.25%

========== TOP 10 WORST WER ==========

5322_7680_000061_000006.wav
WER: 1.0000
CER: 0.4615

GROUND TRUTH:
'Eh, humbug!' thought Lukashka.

PREDICTION:
Hamburg, Dalt Lukaschka,
5322_7679_000024_000000.wav
WER: 0.7143
CER: 0.3611

GROUND TRUTH:
'It's just a habit,' answered Olenin. 'Why?'

PREDICTION:
it's just a habit, and so it all ended. Why?
7800_283493_000070_000000.wav
WER: 0.4000
CER: 0.1000

GROUND TRUTH:
"Got everything now?" asked Bluff.

PREDICTION:
God at everything now? asked Bluff.
6209_34601_000042_000002.wav
WER: 0.4000
CER: 0.0606

GROUND TRUTH:
The savoury odour was perceptible.

PREDICTION:
the savory odor

,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,Since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,0.000000,0.000000
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...",for the same reason. No attention has been giv...,for the same reason no attention has been give...,for the same reason no attention has been give...,0.000000,0.000000
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,readers of these pages will find nothing about...,readers of these pages will find nothing about...,readers of these pages will find nothing about...,0.000000,0.000000
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",part of Chapter 4 has already appeared in the ...,part of chapter iv has already appeared in the...,part of chapter 4 has already appeared in the ...,0.061224,0.021944
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,if Mrs. Shaw had guessed at the real reason wh...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed at the real reason why...,0.014286,0.005263
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...","""'Does he so?' said the Queen, now comprehendi...",does he so said the queen now comprehending all,does he so said the queen now comprehending all,0.000000,0.000000
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",than if you will promise tonight to keep the o...,then if you will promise to night to keep the ...,than if you will promise tonight to keep the o...,0.088235,0.012048
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...","What have I done to thee, that thou shouldst f...",what have i done to thee that thou shouldst fo...,what have i done to thee that thou shouldst fo...,0.037037,0.007407
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","then he rose up, dressed himself hastily, and ...",then he rose up dressed himself hastily and we...,then he rose up dressed himself hastily and we...,0.000000,0.000000


In [ ]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/evc_test_16k/david/david_evc_angry",
    output_csv="evc_david_angry_metrics.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [03:33<00:00,  1.87it/s]


Saved CSV:
evc_david_angry_metrics.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.1231
Median : 0.0909
Std    : 0.1458
Min    : 0.0000
Max    : 1.0000

----- CER -----
Mean   : 0.0665
Median : 0.0399
Std    : 0.0901
Min    : 0.0000
Max    : 0.7317

----- QUALITY -----
Perfect WER: 27.00%
WER <= 0.10: 54.75%
WER <= 0.20: 81.25%
CER <= 0.10: 77.00%

========== TOP 10 WORST WER ==========

5322_7680_000061_000006.wav
WER: 1.0000
CER: 0.5000

GROUND TRUTH:
'Eh, humbug!' thought Lukashka.

PREDICTION:
Air Hamburg. Dr. Lukaszka.
5322_7680_000028_000001.wav
WER: 0.8571
CER: 0.3696

GROUND TRUTH:
Bigger than ours?' asked Lukashka good-naturedly.

PREDICTION:
It's bigger than ours. Ask Luke Ashkagan, mate, Trebly.
6209_34600_000004_000003.wav
WER: 0.8000
CER: 0.5714

GROUND TRUTH:
These honours, however, were deserved.

PREDICTION:
Owners, however, or deserved owners,
7800_283478_000011_000001.wav
WER: 0.7647
CER: 0.7317

GROUND TRUTH:
I happened to dodge a ball fired from the 

,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,0.126984,0.147799
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...","For the same reason, no attention has been giv...",for the same reason no attention has been give...,for the same reason no attention has been give...,0.186047,0.100346
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,His readers of these pages will find nothing a...,readers of these pages will find nothing about...,his readers of these pages will find nothing a...,0.130435,0.082278
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",Part of Chapter 4 has already appeared in the ...,part of chapter iv has already appeared in the...,part of chapter 4 has already appeared in the ...,0.183673,0.156740
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed at the real reason why...,0.300000,0.197368
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...","""'Do as ye so,' said the Queen, now comprehend...",does he so said the queen now comprehending all,do as ye so said the queen now comprehending all,0.333333,0.063830
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",Few will promise to-night to keep the opium-cu...,then if you will promise to night to keep the ...,few will promise to night to keep the opium cu...,0.088235,0.060241
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...","Would have I done to thee, that thou should fo...",what have i done to thee that thou shouldst fo...,would have i done to thee that thou should for...,0.074074,0.044444
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","Then he rose up, dressed himself hastily, and ...",then he rose up dressed himself hastily and we...,then he rose up dressed himself hastily and we...,0.000000,0.000000


In [ ]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/evc_test_16k/david/david_evc_sad",
    output_csv="evc_david_sad_metrics.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [03:27<00:00,  1.93it/s]


Saved CSV:
evc_david_sad_metrics.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.1634
Median : 0.1304
Std    : 0.1585
Min    : 0.0000
Max    : 1.0000

----- CER -----
Mean   : 0.0975
Median : 0.0679
Std    : 0.1050
Min    : 0.0000
Max    : 0.8788

----- QUALITY -----
Perfect WER: 18.00%
WER <= 0.10: 40.75%
WER <= 0.20: 71.50%
CER <= 0.10: 62.00%

========== TOP 10 WORST WER ==========

5322_7680_000061_000006.wav
WER: 1.0000
CER: 0.5000

GROUND TRUTH:
'Eh, humbug!' thought Lukashka.

PREDICTION:
Hamburg. Dr. Lukaszka.
250_142276_000006_000004.wav
WER: 0.8571
CER: 0.6061

GROUND TRUTH:
Her out-of-doors life was perfect.

PREDICTION:
for an archonaut of those perfect.
6209_34601_000156_000001.wav
WER: 0.8571
CER: 0.8788

GROUND TRUTH:
He was answered by a loving growl.

PREDICTION:
out through a loving growl, through well.
5322_7679_000002_000003.wav
WER: 0.8000
CER: 0.4688

GROUND TRUTH:
The Cossacks received him coldly.

PREDICTION:
Bacassex received from Cobley.
6209_3

,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,0.190476,0.179245
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...",the same reason. No attention has been given t...,for the same reason no attention has been give...,the same reason no attention has been given to...,0.162791,0.138408
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,readers of these pages will find nothing about...,readers of these pages will find nothing about...,readers of these pages will find nothing about...,0.130435,0.056962
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",Chapter 5 of Chapter 4 has already appeared in...,part of chapter iv has already appeared in the...,chapter 5 of chapter 4 has already appeared in...,0.367347,0.235110
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,if Mrs. Shaw had guessed at the real reason wh...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed at the real reason why...,0.185714,0.150000
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...",so said the Queen now comprehending all,does he so said the queen now comprehending all,so said the queen now comprehending all,0.222222,0.170213
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",than if you will promise tonight to keep the o...,then if you will promise to night to keep the ...,than if you will promise tonight to keep the o...,0.264706,0.102410
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...","what have I done to thee, that thou shouldst f...",what have i done to thee that thou shouldst fo...,what have i done to thee that thou shouldst fo...,0.000000,0.000000
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","then he rose up, dressed himself hastily, and ...",then he rose up dressed himself hastily and we...,then he rose up dressed himself hastily and we...,0.000000,0.000000


In [ ]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/test_16k/vc_morgan",
    output_csv="/content/vc_morgan_results.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [03:43<00:00,  1.79it/s]


Saved CSV:
/content/vc_morgan_results.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.0416
Median : 0.0000
Std    : 0.0906
Min    : 0.0000
Max    : 1.0000

----- CER -----
Mean   : 0.0163
Median : 0.0000
Std    : 0.0412
Min    : 0.0000
Max    : 0.4615

----- QUALITY -----
Perfect WER: 63.75%
WER <= 0.10: 86.00%
WER <= 0.20: 95.25%
CER <= 0.10: 96.50%

========== TOP 10 WORST WER ==========

5322_7680_000061_000006.wav
WER: 1.0000
CER: 0.4615

GROUND TRUTH:
'Eh, humbug!' thought Lukashka.

PREDICTION:
Herr Hamburg, Herr Lukaszka.
8312_279790_000026_000000.wav
WER: 0.5000
CER: 0.1154

GROUND TRUTH:
"Am not I she?" said Troutina.

PREDICTION:
I'm not, I see, said Trotina.
6209_34601_000111_000000.wav
WER: 0.4545
CER: 0.2542

GROUND TRUTH:
Ursus seized the pitcher again, and conveyed it to his mouth.

PREDICTION:
Arsas sees the picture again and can bait it to his mouth.
5322_7680_000028_000001.wav
WER: 0.4286
CER: 0.1957

GROUND TRUTH:
Bigger than ours?' asked Lukashka goo

,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,since my subject is not the splendor of a hist...,since my subject is not the splendor of histor...,since my subject is not the splendor of a hist...,0.015873,0.006289
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...",for the same reason no attention has been give...,for the same reason no attention has been give...,for the same reason no attention has been give...,0.093023,0.079585
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,Readers of these pages will find nothing about...,readers of these pages will find nothing about...,readers of these pages will find nothing about...,0.000000,0.000000
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",Part of Chapter 4 has already appeared in the ...,part of chapter iv has already appeared in the...,part of chapter 4 has already appeared in the ...,0.061224,0.021944
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed at the real reason why...,0.014286,0.005263
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...","Does he so? said the Queen, now comprehending ...",does he so said the queen now comprehending all,does he so said the queen now comprehending all,0.000000,0.000000
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",then if you will promise to-night to keep the ...,then if you will promise to night to keep the ...,then if you will promise to night to keep the ...,0.000000,0.000000
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...","What have I done to thee, that thou shouldst f...",what have i done to thee that thou shouldst fo...,what have i done to thee that thou shouldst fo...,0.037037,0.007407
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","Then he rose up, dressed himself hastily, and ...",then he rose up dressed himself hastily and we...,then he rose up dressed himself hastily and we...,0.000000,0.000000


In [ ]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/evc_test_16k/morgan/morgan_evc_angry",
    output_csv="evc_morgan_angry_metrics.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [03:30<00:00,  1.90it/s]


Saved CSV:
evc_morgan_angry_metrics.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.1106
Median : 0.0714
Std    : 0.1424
Min    : 0.0000
Max    : 1.0000

----- CER -----
Mean   : 0.0526
Median : 0.0295
Std    : 0.0712
Min    : 0.0000
Max    : 0.5238

----- QUALITY -----
Perfect WER: 31.00%
WER <= 0.10: 63.00%
WER <= 0.20: 83.75%
CER <= 0.10: 83.50%

========== TOP 10 WORST WER ==========

5322_7680_000061_000006.wav
WER: 1.0000
CER: 0.4615

GROUND TRUTH:
'Eh, humbug!' thought Lukashka.

PREDICTION:
Air Hamburg, St. Lukaska.
6209_34600_000009_000001.wav
WER: 0.8750
CER: 0.5238

GROUND TRUTH:
Now Melcombe Regis is a parish of Weymouth.

PREDICTION:
Now I'm going to come read jazz. This is a parish of Weymouth.
5322_7679_000002_000003.wav
WER: 0.8000
CER: 0.3750

GROUND TRUTH:
The Cossacks received him coldly.

PREDICTION:
The Cossacks receive Tim Kuzel, please.
5322_7680_000028_000001.wav
WER: 0.7143
CER: 0.3696

GROUND TRUTH:
Bigger than ours?' asked Lukashka good-nature

,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,Since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,0.015873,0.009434
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...","For the same reason, no attention has been giv...",for the same reason no attention has been give...,for the same reason no attention has been give...,0.046512,0.010381
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,Readers of these pages will fend nothing about...,readers of these pages will find nothing about...,readers of these pages will fend nothing about...,0.086957,0.025316
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",Part of Chapter 4 has already appeared in the ...,part of chapter iv has already appeared in the...,part of chapter 4 has already appeared in the ...,0.061224,0.021944
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,If Mrs. Shaw had guessed that the real reason ...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed that the real reason w...,0.128571,0.068421
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...","How does he so, said the queen, not comprehend...",does he so said the queen now comprehending all,how does he so said the queen not comprehendin...,0.222222,0.106383
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",Then if you will promise tonight to keep the O...,then if you will promise to night to keep the ...,then if you will promise tonight to keep the o...,0.088235,0.018072
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...","What have I done to thee, that thou shouldst f...",what have i done to thee that thou shouldst fo...,what have i done to thee that thou shouldst fo...,0.037037,0.029630
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","Then he rose up, dressed himself hastily, and ...",then he rose up dressed himself hastily and we...,then he rose up dressed himself hastily and we...,0.000000,0.000000


In [ ]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/evc_test_16k/morgan/morgan_evc_sad",
    output_csv="evc_morgan_sad_metrics.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [03:26<00:00,  1.94it/s]


Saved CSV:
evc_morgan_sad_metrics.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.1437
Median : 0.0814
Std    : 0.1982
Min    : 0.0000
Max    : 1.0000

----- CER -----
Mean   : 0.0938
Median : 0.0365
Std    : 0.1626
Min    : 0.0000
Max    : 0.8491

----- QUALITY -----
Perfect WER: 29.25%
WER <= 0.10: 60.25%
WER <= 0.20: 78.50%
CER <= 0.10: 76.25%

========== TOP 10 WORST WER ==========

6209_34600_000007_000002.wav
WER: 1.0000
CER: 0.8333

GROUND TRUTH:
At intervals he knocked at the doors.

PREDICTION:
Thank you.
6209_34599_000018_000002.wav
WER: 1.0000
CER: 0.7826

GROUND TRUTH:
Roofs--dwellings--shelter!

PREDICTION:
CHALTER
6209_34599_000009_000000.wav
WER: 1.0000
CER: 0.8333

GROUND TRUTH:
Two or three times the little infant cried.

PREDICTION:
Illich and Pankright
5322_7678_000006_000026.wav
WER: 0.9444
CER: 0.8293

GROUND TRUTH:
He thought of God and of the future life as for long he had not thought about them.

PREDICTION:
Out for the bell.
78_369_000069_000010

,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,0.000000,0.000000
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...","For the same reason, no attention has been giv...",for the same reason no attention has been give...,for the same reason no attention has been give...,0.023256,0.020761
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,Readers of these pages will find nothing about...,readers of these pages will find nothing about...,readers of these pages will find nothing about...,0.043478,0.031646
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",Part of Chapter 4 has already appeared in the ...,part of chapter iv has already appeared in the...,part of chapter 4 has already appeared in the ...,0.081633,0.062696
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,if mrs shaw had guessed that the real reason w...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed that the real reason w...,0.185714,0.142105
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...","This is so, said the Queen, now comprehending ...",does he so said the queen now comprehending all,this is so said the queen now comprehending all,0.222222,0.106383
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",then if you will promise to-night to keep the ...,then if you will promise to night to keep the ...,then if you will promise to night to keep the ...,0.029412,0.018072
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...",that thou should forget me and marry Troutina ...,what have i done to thee that thou shouldst fo...,that thou should forget me and marry troutina ...,0.333333,0.281481
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","Then he rose up, dressed himself hastily, and ...",then he rose up dressed himself hastily and we...,then he rose up dressed himself hastily and we...,0.000000,0.000000
